1.Requirements

In [ ]:
!git clone https://github.com/Loping151/EndoGSLAM.git
%cd EndoGSLAM

# Fix: the repo has _init_.py (single underscores) instead of __init__.py
# Python requires __init__.py to recognize datasets/ as a package
!cp datasets/_init_.py datasets/__init__.py

In [ ]:
!nvcc --version

In [ ]:
!pip uninstall torch torchvision torchaudio -y
!pip install torch torchvision torchaudio

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")

In [ ]:
%%writefile requirements.txt
tqdm>=4.65.0
numpy>=1.24.0
Pillow>=9.2.0
opencv-python>=4.9.0
imageio
matplotlib>=3.5.2
kornia
natsort
pyyaml
plotly
lpips
open3d>=0.16.0
torchmetrics
# cyclonedds  # Skipping - build issues in Colab
pytorch-msssim
jupyter
trimesh
# git+https://github.com/JonathonLuiten/diff-gaussian-rasterization-w-depth/  # Installed separately after patching

In [ ]:
!apt-get update
!apt-get install -y build-essential
!apt-get install -y cuda-toolkit-12-5

In [ ]:
!pip install -r requirements.txt

2. Patch CUDA Rasterizer for Per-Gaussian Visibility

In [ ]:
# First, clone the rasterizer repo separately (it is NOT a submodule in EndoGSLAM)
%cd /content
!git clone https://github.com/JonathonLuiten/diff-gaussian-rasterization-w-depth.git

import os

RAST_DIR = "/content/diff-gaussian-rasterization-w-depth"

# ============================
# 1. cuda_rasterizer/forward.h
# ============================
fpath = os.path.join(RAST_DIR, "cuda_rasterizer/forward.h")
with open(fpath, "r") as f:
    src = f.read()

# Simple targeted replacement: add out_gaussian_visibility to render declaration
src = src.replace(
    'float* out_depth);',
    'float* out_depth,\n\t\tfloat* out_gaussian_visibility);'
)
with open(fpath, 'w') as f:
    f.write(src)
print(f'[PATCHED] {fpath}')

# ============================
# 2. cuda_rasterizer/forward.cu
# ============================
fpath = os.path.join(RAST_DIR, "cuda_rasterizer/forward.cu")
with open(fpath, "r") as f:
    src = f.read()

# 2a. Add out_gaussian_visibility to renderCUDA kernel signature
src = src.replace(
    'float* __restrict__ out_depth)',
    'float* __restrict__ out_depth,\n\tfloat* __restrict__ out_gaussian_visibility)'
)

# 2b. Add atomicAdd after color accumulation in the inner loop
# The actual code uses: features[collected_id[j] * CHANNELS + ch] * alpha * T
src = src.replace(
    'for (int ch = 0; ch < CHANNELS; ch++)\n\t\t\t\tC[ch] += features[collected_id[j] * CHANNELS + ch] * alpha * T;',
    'for (int ch = 0; ch < CHANNELS; ch++)\n\t\t\t\tC[ch] += features[collected_id[j] * CHANNELS + ch] * alpha * T;\n\n\t\t\t\t// Per-Gaussian visibility accumulation (Innovation 1)\n\t\t\t\tif (inside)\n\t\t\t\t\tatomicAdd(&out_gaussian_visibility[collected_id[j]], alpha * T);'
)

# 2c. Update FORWARD::render wrapper signature and kernel launch
# Signature
src = src.replace(
    'const float* depth,\n\tfloat* out_depth)\n{\n\trenderCUDA<NUM_CHANNELS>',
    'const float* depth,\n\tfloat* out_depth,\n\tfloat* out_gaussian_visibility)\n{\n\trenderCUDA<NUM_CHANNELS>'
)
# Kernel call arguments
src = src.replace(
    'depth,\n\t\tout_depth);',
    'depth,\n\t\tout_depth,\n\t\tout_gaussian_visibility);'
)

with open(fpath, 'w') as f:
    f.write(src)
print(f'[PATCHED] {fpath}')

# ============================
# 3. cuda_rasterizer/rasterizer.h
# ============================
fpath = os.path.join(RAST_DIR, 'cuda_rasterizer/rasterizer.h')
with open(fpath, 'r') as f:
    src = f.read()
# Add out_gaussian_visibility parameter with default nullptr
src = src.replace(
    'int* radii = nullptr);',
    'int* radii = nullptr,\n\t\t\tfloat* out_gaussian_visibility = nullptr);'
)
with open(fpath, 'w') as f:
    f.write(src)
print(f'[PATCHED] {fpath}')

# ============================
# 4. cuda_rasterizer/rasterizer_impl.cu
# ============================
fpath = os.path.join(RAST_DIR, 'cuda_rasterizer/rasterizer_impl.cu')
with open(fpath, 'r') as f:
    src = f.read()
# 4a. Update forward() signature
src = src.replace(
    'float* out_depth,\n\tint* radii)',
    'float* out_depth,\n\tint* radii,\n\tfloat* out_gaussian_visibility)'
)
# 4b. Pass to FORWARD::render call
src = src.replace(
    'geomState.depths,\n\t\tout_depth);',
    'geomState.depths,\n\t\tout_depth,\n\t\tout_gaussian_visibility);'
)
with open(fpath, 'w') as f:
    f.write(src)
print(f'[PATCHED] {fpath}')

# ============================
# 5. rasterize_points.cu
# ============================
fpath = os.path.join(RAST_DIR, 'rasterize_points.cu')
with open(fpath, 'r') as f:
    src = f.read()
# 5a. Update return type (add one more torch::Tensor)
src = src.replace(
    'std::tuple<int, torch::Tensor, torch::Tensor, torch::Tensor, torch::Tensor, torch::Tensor, torch::Tensor>\nRasterizeGaussiansCUDA',
    'std::tuple<int, torch::Tensor, torch::Tensor, torch::Tensor, torch::Tensor, torch::Tensor, torch::Tensor, torch::Tensor>\nRasterizeGaussiansCUDA'
)
# 5b. Allocate out_gaussian_visibility tensor
src = src.replace(
    'torch::Tensor out_depth = torch::full({1, H, W}, 0.0, float_opts);',
    'torch::Tensor out_depth = torch::full({1, H, W}, 0.0, float_opts);\n  torch::Tensor out_gaussian_visibility = torch::zeros({P}, float_opts);'
)
# 5c. Pass to Rasterizer::forward
src = src.replace(
    'radii.contiguous().data<int>());',
    'radii.contiguous().data<int>(),\n\t\tout_gaussian_visibility.contiguous().data<float>());'
)
# 5d. Update return tuple
src = src.replace(
    'return std::make_tuple(rendered, out_color, radii, geomBuffer, binningBuffer, imgBuffer, out_depth);',
    'return std::make_tuple(rendered, out_color, radii, geomBuffer, binningBuffer, imgBuffer, out_depth, out_gaussian_visibility);'
)
with open(fpath, 'w') as f:
    f.write(src)
print(f'[PATCHED] {fpath}')

# ============================
# 6. diff_gaussian_rasterization/__init__.py
# ============================
fpath = os.path.join(RAST_DIR, 'diff_gaussian_rasterization/__init__.py')
with open(fpath, 'r') as f:
    src = f.read()
# 6a. Unpack 8 values from C++ call
src = src.replace(
    'num_rendered, color, radii, geomBuffer, binningBuffer, imgBuffer, depth = _C.rasterize_gaussians(*args)',
    'num_rendered, color, radii, geomBuffer, binningBuffer, imgBuffer, depth, gaussian_visibility = _C.rasterize_gaussians(*args)'
)
# 6b. Return 4 values
src = src.replace(
    'return color, radii, depth',
    'return color, radii, depth, gaussian_visibility'
)
# 6c. Update backward to accept 4 grad inputs
src = src.replace(
    'def backward(ctx, grad_out_color, _, depth):',
    'def backward(ctx, grad_out_color, _, depth, __):'
)
with open(fpath, 'w') as f:
    f.write(src)
print(f'[PATCHED] {fpath}')

# ============================
# 7. rasterize_points.h — header must match .cu return type
# ============================
fpath = os.path.join(RAST_DIR, 'rasterize_points.h')
with open(fpath, 'r') as f:
    src = f.read()
src = src.replace(
    'std::tuple<int, torch::Tensor, torch::Tensor, torch::Tensor, torch::Tensor, torch::Tensor, torch::Tensor>\nRasterizeGaussiansCUDA',
    'std::tuple<int, torch::Tensor, torch::Tensor, torch::Tensor, torch::Tensor, torch::Tensor, torch::Tensor, torch::Tensor>\nRasterizeGaussiansCUDA'
)
with open(fpath, 'w') as f:
    f.write(src)
print(f'[PATCHED] {fpath}')

print('\n=== All 7 files patched successfully! ===')

In [ ]:
%cd /content/diff-gaussian-rasterization-w-depth
!pip install .
%cd /content/EndoGSLAM

3. Prepare Data

In [ ]:
# Install gdown
!pip install gdown

# Create data directory
!mkdir -p data/C3VD

# Download dataset from Google Drive
!gdown 1MwpfFKKweM1L3bYY7V4dYzS_IlkFaOUe -O /content/C3VD_EndoGSLAM.tar.gz

# Extract dataset
!tar -xzf /content/C3VD_EndoGSLAM.tar.gz -C data/

# Check data directory structure
!ls -la data/C3VD/

In [ ]:
# Verify data structure - check one scene
!ls -la data/C3VD/cecum_t1_b/

# Check for color, depth folders and pose.txt
!ls data/C3VD/cecum_t1_b/color/ | head -5
!ls data/C3VD/cecum_t1_b/depth/ | head -5
!cat data/C3VD/cecum_t1_b/pose.txt | head -5

# Switch to EndoGSLAM directory
%cd /content/EndoGSLAM

# Check config files
!ls configs/c3vd/

In [ ]:
# Fix ALL Renderer() calls in the codebase to unpack 4 values instead of 3
# (our patched rasterizer now returns color, radii, depth, gaussian_visibility)
import glob, os

files_to_fix = [
    'utils/eval_helpers.py',
    'utils/online_render.py',
    'viz_scripts/online_recon.py',
    'viz_scripts/final_recon.py',
]

for fname in files_to_fix:
    fpath = os.path.join('/content/EndoGSLAM', fname)
    if not os.path.exists(fpath):
        print(f'[SKIP] {fpath} not found')
        continue
    with open(fpath, 'r') as f:
        src = f.read()
    original = src
    # All patterns of 3-value Renderer unpacking
    src = src.replace('depth_sil, _, _ = Renderer(', 'depth_sil, _, _, _ = Renderer(')
    src = src.replace('im, radius, _ = Renderer(', 'im, radius, _, _ = Renderer(')
    src = src.replace('im, _, depth = Renderer(', 'im, _, depth, _ = Renderer(')
    src = src.replace('im, _, _ = Renderer(', 'im, _, _, _ = Renderer(')
    src = src.replace('im, _, _= Renderer(', 'im, _, _, _= Renderer(')
    if src != original:
        with open(fpath, 'w') as f:
            f.write(src)
        print(f'[FIXED] {fname}')
    else:
        print(f'[OK] {fname} - no changes needed')

print('\nAll Renderer calls updated.')

4. Apply Innovation Code

In [ ]:
%%writefile configs/c3vd/c3vd_innovations.py
import os

scenes = [
    "cecum_t1_b", 
    "cecum_t2_b", 
    "cecum_t3_a", 
    "sigmoid_t1_a", 
    "sigmoid_t2_a", 
    "sigmoid_t3_a", 
    "trans_t1_b", 
    "trans_t2_c", 
    "trans_t4_a", 
    "trans_t4_b"
]

primary_device="cuda:0"
seed = 0
try:    
    scene_name = scenes[int(os.environ["SCENE_NUM"])]
except KeyError:
    scene_name = "sigmoid_t3_a"

map_every = 1
keyframe_every = 8
tracking_iters = 15
mapping_iters = 25

group_name = "C3VD_innovations"
run_name = scene_name

config = dict(
    workdir=f"./experiments/{group_name}",
    run_name=run_name,
    seed=seed,
    primary_device=primary_device,
    map_every=map_every,
    keyframe_every=keyframe_every,
    distance_keyframe_selection=True,
    distance_current_frame_prob=0.1,
    mapping_window_size=-1,
    report_global_progress_every=2000,
    scene_radius_depth_ratio=3,
    mean_sq_dist_method="projective",
    report_iter_progress=False,
    load_checkpoint=False,
    checkpoint_time_idx=0,
    save_checkpoints=False,
    checkpoint_interval=int(1e10),
    gaussian_simplification=True,
    data=dict(
        basedir="./data/C3VD",
        gradslam_data_cfg="./configs/data/c3vd.yaml",
        sequence=scene_name,
        desired_image_height=1080//2,
        desired_image_width=1350//2,
        start=0,
        end=-1,
        stride=1,
        num_frames=-1,
        train_or_test="train",
    ),
    tracking=dict(
        use_gt_poses=False,
        forward_prop=True,
        num_iters=tracking_iters,
        use_sil_for_loss=True,
        sil_thres=0.99,
        use_l1=True,
        ignore_outlier_depth_loss=False,
        loss_weights=dict(
            im=0.5,
            depth=1.0,
        ),
        lrs=dict(
            means3D=0.0,
            rgb_colors=0.0,
            unnorm_rotations=0.0,
            logit_opacities=0.0,
            log_scales=0.0,
            cam_unnorm_rots=0.002,
            cam_trans=0.005,
        ),
    ),
    mapping=dict(
        num_iters=mapping_iters,
        add_new_gaussians=True,
        sil_thres=0.5,
        use_l1=True,
        use_sil_for_loss=False,
        ignore_outlier_depth_loss=False,
        loss_weights=dict(
            im=1.0,
            depth=1.0,
        ),
        lrs=dict(
            means3D=0.0001,
            rgb_colors=0.0025,
            unnorm_rotations=0.001,
            logit_opacities=0.05,
            log_scales=0.001,
            cam_unnorm_rots=0.000,
            cam_trans=0.000,
        ),
        prune_gaussians=True,
        pruning_dict=dict(
            start_after=0,
            remove_big_after=0,
            stop_after=20,
            prune_every=20,
            removal_opacity_threshold=0.005,
            final_removal_opacity_threshold=0.005,
            reset_opacities=False,
            reset_opacities_every=int(1e10),
        ),
        use_gaussian_splatting_densification=False,
        densify_dict=dict(
            start_after=500,
            remove_big_after=3000,
            stop_after=5000,
            densify_every=100,
            grad_thresh=0.0002,
            num_to_split_into=2,
            removal_opacity_threshold=0.005,
            final_removal_opacity_threshold=0.005,
            reset_opacities_every=3000,
        ),
    ),
    # ============================================
    # INNOVATION CONFIG - Toggle each independently
    # ============================================
    innovations=dict(
        # --- Innovation 1: Visibility-aware dual-mask pruning ---
        enable_visibility_pruning=True,
        distance_gamma=0.05,         # Depth diff threshold for floater detection
        degeneration_eta=0.2,        # Opacity degeneration factor
        vis_threshold=0.3,           # Min visibility ratio
        min_observations=5,          # Min frames before visibility pruning kicks in
        vis_window_size=15,          # Circular buffer size for visibility history

        # --- Innovation 2: Periodic Bundle Adjustment ---
        enable_periodic_ba=True,
        ba_every_m_frames=20,        # BA trigger frequency
        ba_n_keyframes=5,            # Number of keyframes per BA
        ba_num_iters=30,             # BA optimization iterations
        ba_selection='hybrid',       # Keyframe selection: 'random', 'recent', 'hybrid'
        ba_lrs=dict(
            means3D=0.00005,
            rgb_colors=0.001,
            unnorm_rotations=0.0005,
            logit_opacities=0.025,
            log_scales=0.0005,
            cam_unnorm_rots=0.001,   # NON-ZERO: this is the key difference from mapping
            cam_trans=0.002,          # NON-ZERO: this is the key difference from mapping
        ),

        # --- Innovation 3: Deformation modeling ---
        enable_deformation=True,
        deform_lr=0.0005,            # Learning rate for deformation offsets
        var_threshold=0.1,           # Visibility variance threshold for deformation detection
        lambda_deform_temporal=0.1,  # Temporal smoothness weight
        lambda_deform_magnitude=0.01,# Magnitude penalty weight

        # --- Tracking with deformation confidence (down-weight deforming regions) ---
        enable_deform_weighted_tracking=True,
    ),
    viz=dict(
        render_mode='color',
        offset_first_viz_cam=True,
        show_sil=False,
        visualize_cams=False,
        viz_w=320, viz_h=320,
        viz_near=0.01, viz_far=100.0,
        view_scale=2,
        viz_fps=30,
        enter_interactive_post_online=True,
        gaussian_simplification=False,
    ),
)

In [ ]:
%%writefile utils/slam_external.py
"""
# Copyright (C) 2023, Inria
# GRAPHDECO research group, https://team.inria.fr/graphdeco
# All rights reserved.
#
# This software is free for non-commercial, research and evaluation use
# under the terms of the LICENSE.md file found here:
# https://github.com/graphdeco-inria/gaussian-splatting/blob/main/LICENSE.md
#
# For inquiries contact  george.drettakis@inria.fr

#######################################################################################################################
##### NOTE: CODE IN THIS FILE IS NOT INCLUDED IN THE OVERALL PROJECT'S MIT LICENSE #####
##### USE OF THIS CODE FOLLOWS THE COPYRIGHT NOTICE ABOVE #####
#######################################################################################################################
"""
import numpy as np
import torch
import torch.nn.functional as func
from torch.autograd import Variable
from math import exp


def build_rotation(q):
    norm = torch.sqrt(q[:, 0] * q[:, 0] + q[:, 1] * q[:, 1] + q[:, 2] * q[:, 2] + q[:, 3] * q[:, 3])
    q = q / norm[:, None]
    rot = torch.zeros((q.size(0), 3, 3), device='cuda')
    r = q[:, 0]
    x = q[:, 1]
    y = q[:, 2]
    z = q[:, 3]
    rot[:, 0, 0] = 1 - 2 * (y * y + z * z)
    rot[:, 0, 1] = 2 * (x * y - r * z)
    rot[:, 0, 2] = 2 * (x * z + r * y)
    rot[:, 1, 0] = 2 * (x * y + r * z)
    rot[:, 1, 1] = 1 - 2 * (x * x + z * z)
    rot[:, 1, 2] = 2 * (y * z - r * x)
    rot[:, 2, 0] = 2 * (x * z - r * y)
    rot[:, 2, 1] = 2 * (y * z + r * x)
    rot[:, 2, 2] = 1 - 2 * (x * x + y * y)
    return rot


def calc_mse(img1, img2):
    return ((img1 - img2) ** 2).view(img1.shape[0], -1).mean(1, keepdim=True)


def calc_psnr(img1, img2):
    mse = ((img1 - img2) ** 2).view(img1.shape[0], -1).mean(1, keepdim=True)
    return 20 * torch.log10(1.0 / torch.sqrt(mse))


def gaussian(window_size, sigma):
    gauss = torch.Tensor([exp(-(x - window_size // 2) ** 2 / float(2 * sigma ** 2)) for x in range(window_size)])
    return gauss / gauss.sum()


def create_window(window_size, channel):
    _1D_window = gaussian(window_size, 1.5).unsqueeze(1)
    _2D_window = _1D_window.mm(_1D_window.t()).float().unsqueeze(0).unsqueeze(0)
    window = Variable(_2D_window.expand(channel, 1, window_size, window_size).contiguous())
    return window


def calc_ssim(img1, img2, window_size=11, size_average=True):
    channel = img1.size(-3)
    window = create_window(window_size, channel)
    if img1.is_cuda:
        window = window.cuda(img1.get_device())
    window = window.type_as(img1)
    return _ssim(img1, img2, window, window_size, channel, size_average)


def _ssim(img1, img2, window, window_size, channel, size_average=True):
    mu1 = func.conv2d(img1, window, padding=window_size // 2, groups=channel)
    mu2 = func.conv2d(img2, window, padding=window_size // 2, groups=channel)
    mu1_sq = mu1.pow(2)
    mu2_sq = mu2.pow(2)
    mu1_mu2 = mu1 * mu2
    sigma1_sq = func.conv2d(img1 * img1, window, padding=window_size // 2, groups=channel) - mu1_sq
    sigma2_sq = func.conv2d(img2 * img2, window, padding=window_size // 2, groups=channel) - mu2_sq
    sigma12 = func.conv2d(img1 * img2, window, padding=window_size // 2, groups=channel) - mu1_mu2
    c1 = 0.01 ** 2
    c2 = 0.03 ** 2
    ssim_map = ((2 * mu1_mu2 + c1) * (2 * sigma12 + c2)) / ((mu1_sq + mu2_sq + c1) * (sigma1_sq + sigma2_sq + c2))
    if size_average:
        return ssim_map.mean()
    else:
        return ssim_map.mean(1).mean(1).mean(1)


def accumulate_mean2d_gradient(variables):
    variables['means2D_gradient_accum'][variables['seen']] += torch.norm(
        variables['means2D'].grad[variables['seen'], :2], dim=-1)
    variables['denom'][variables['seen']] += 1
    return variables


def update_params_and_optimizer(new_params, params, optimizer):
    for k, v in new_params.items():
        group = [x for x in optimizer.param_groups if x["name"] == k][0]
        stored_state = optimizer.state.get(group['params'][0], None)
        stored_state["exp_avg"] = torch.zeros_like(v)
        stored_state["exp_avg_sq"] = torch.zeros_like(v)
        del optimizer.state[group['params'][0]]
        group["params"][0] = torch.nn.Parameter(v.requires_grad_(True))
        optimizer.state[group['params'][0]] = stored_state
        params[k] = group["params"][0]
    return params


def cat_params_to_optimizer(new_params, params, optimizer):
    for k, v in new_params.items():
        group = [g for g in optimizer.param_groups if g['name'] == k][0]
        stored_state = optimizer.state.get(group['params'][0], None)
        if stored_state is not None:
            stored_state["exp_avg"] = torch.cat((stored_state["exp_avg"], torch.zeros_like(v)), dim=0)
            stored_state["exp_avg_sq"] = torch.cat((stored_state["exp_avg_sq"], torch.zeros_like(v)), dim=0)
            del optimizer.state[group['params'][0]]
            group["params"][0] = torch.nn.Parameter(torch.cat((group["params"][0], v), dim=0).requires_grad_(True))
            optimizer.state[group['params'][0]] = stored_state
            params[k] = group["params"][0]
        else:
            group["params"][0] = torch.nn.Parameter(torch.cat((group["params"][0], v), dim=0).requires_grad_(True))
            params[k] = group["params"][0]
    return params


def _resize_1d(tensor, N, dtype=None):
    """Resize a 1D tensor to length N: pad with zeros or truncate."""
    old_N = tensor.shape[0]
    if old_N == N:
        return tensor
    elif old_N < N:
        pad = torch.zeros(N - old_N, device=tensor.device, dtype=dtype or tensor.dtype)
        return torch.cat([tensor, pad])
    else:
        return tensor[:N]


def _resize_to_N(tensor, N):
    """Resize a 2D tensor [old_N, D] to [N, D]: pad with zeros or truncate."""
    old_N = tensor.shape[0]
    if old_N == N:
        return tensor
    elif old_N < N:
        pad_shape = list(tensor.shape)
        pad_shape[0] = N - old_N
        pad = torch.zeros(pad_shape, device=tensor.device, dtype=tensor.dtype)
        return torch.cat([tensor, pad], dim=0)
    else:
        return tensor[:N]


def remove_points(to_remove, params, variables, optimizer):
    to_keep = ~to_remove
    keys = [k for k in params.keys() if k not in ['cam_unnorm_rots', 'cam_trans']]
    for k in keys:
        group = [g for g in optimizer.param_groups if g['name'] == k][0]
        stored_state = optimizer.state.get(group['params'][0], None)
        if stored_state is not None:
            stored_state["exp_avg"] = stored_state["exp_avg"][to_keep]
            stored_state["exp_avg_sq"] = stored_state["exp_avg_sq"][to_keep]
            del optimizer.state[group['params'][0]]
            group["params"][0] = torch.nn.Parameter((group["params"][0][to_keep].requires_grad_(True)))
            optimizer.state[group['params'][0]] = stored_state
            params[k] = group["params"][0]
        else:
            group["params"][0] = torch.nn.Parameter(group["params"][0][to_keep].requires_grad_(True))
            params[k] = group["params"][0]
    variables['means2D_gradient_accum'] = variables['means2D_gradient_accum'][to_keep]
    variables['denom'] = variables['denom'][to_keep]
    variables['max_2D_radius'] = variables['max_2D_radius'][to_keep]
    if 'timestep' in variables.keys():
        variables['timestep'] = variables['timestep'][to_keep]
    # === INNOVATION 1+3: Keep innovation variables in sync ===
    N = to_keep.shape[0]
    if 'vis_history' in variables:
        variables['vis_history'] = _resize_to_N(variables['vis_history'], N)[to_keep]
    if 'vis_frame_count' in variables:
        variables['vis_frame_count'] = _resize_1d(variables['vis_frame_count'], N)[to_keep]
    if 'deform_mask' in variables:
        variables['deform_mask'] = _resize_1d(variables['deform_mask'], N, dtype=torch.bool)[to_keep]
    if 'prev_deform_offsets' in variables:
        variables['prev_deform_offsets'] = _resize_to_N(variables['prev_deform_offsets'], N)[to_keep]
    return params, variables


def inverse_sigmoid(x):
    return torch.log(x / (1 - x))


# ============================================================
# INNOVATION 1: Distance-based floater detection (GS-SLAM Eq.9)
# ============================================================

def compute_distance_mask(transformed_pts, curr_data, gamma=0.05):
    """
    Detect Gaussians floating in front of the observed surface.
    Simplified GS-SLAM Eq. 9: compare Gaussian z-depth to observed depth.

    Args:
        transformed_pts: [N, 3] Gaussian positions in camera frame
        curr_data: dict with 'depth', 'intrinsics'
        gamma: depth difference threshold
    Returns:
        floater_mask: [N] boolean, True = floater
    """
    gauss_depth = transformed_pts[:, 2].detach()  # [N]

    intrinsics = curr_data['intrinsics']
    FX = intrinsics[0, 0]
    FY = intrinsics[1, 1]
    CX = intrinsics[0, 2]
    CY = intrinsics[1, 2]

    pts_cam = transformed_pts.detach()
    u = (FX * pts_cam[:, 0] / (pts_cam[:, 2] + 1e-6) + CX).long()
    v = (FY * pts_cam[:, 1] / (pts_cam[:, 2] + 1e-6) + CY).long()

    H = curr_data['depth'].shape[1]
    W = curr_data['depth'].shape[2]
    u = u.clamp(0, W - 1)
    v = v.clamp(0, H - 1)

    gt_depth = curr_data['depth'].squeeze()  # [H, W]
    observed_depth = gt_depth[v, u]  # [N]

    # Gaussian is in front of surface by more than gamma
    depth_diff = observed_depth - gauss_depth
    floater_mask = depth_diff > gamma

    valid_depth = observed_depth > 0
    valid_gauss = gauss_depth > 0
    floater_mask = floater_mask & valid_depth & valid_gauss

    return floater_mask


# ============================================================
# INNOVATION 1: Enhanced pruning with dual-mask
# ============================================================

def prune_gaussians(params, variables, optimizer, iter, prune_dict,
                    innovation_config=None, curr_data=None, transformed_pts=None):
    """
    Enhanced pruning with Innovation 1: dual-mask (visibility + distance).
    Falls back to original behavior when innovation_config is None.
    """
    if iter <= prune_dict['stop_after']:
        if (iter >= prune_dict['start_after']) and (iter % prune_dict['prune_every'] == 0):
            if iter == prune_dict['stop_after']:
                remove_threshold = prune_dict['final_removal_opacity_threshold']
            else:
                remove_threshold = prune_dict['removal_opacity_threshold']

            # === INNOVATION 1: Distance + Visibility opacity degeneration ===
            if (innovation_config is not None and
                innovation_config.get('enable_visibility_pruning', False) and
                curr_data is not None and transformed_pts is not None):

                gamma = innovation_config.get('distance_gamma', 0.05)
                eta = innovation_config.get('degeneration_eta', 0.2)

                # Distance-based floater detection
                floater_mask = compute_distance_mask(transformed_pts, curr_data, gamma=gamma)

                # Visibility-based floater detection
                if 'vis_history' in variables and 'vis_frame_count' in variables:
                    vis_threshold = innovation_config.get('vis_threshold', 0.3)
                    min_obs = innovation_config.get('min_observations', 5)

                    N = floater_mask.shape[0]
                    vis_hist = _resize_to_N(variables['vis_history'], N)
                    vis_cnt = _resize_1d(variables['vis_frame_count'], N)
                    vis_mean = vis_hist.mean(dim=1)  # [N]
                    low_vis = vis_mean < vis_threshold
                    enough_obs = vis_cnt > min_obs
                    vis_floater_mask = low_vis & enough_obs
                    # Combine: distance OR visibility floater
                    combined_floater = floater_mask | vis_floater_mask
                else:
                    combined_floater = floater_mask

                # Degenerate opacity of floaters (GS-SLAM style)
                with torch.no_grad():
                    if combined_floater.any():
                        opacities = torch.sigmoid(params['logit_opacities'].squeeze())
                        opacities[combined_floater] = opacities[combined_floater] * eta
                        opacities = opacities.clamp(1e-6, 1 - 1e-6)
                        new_logit = torch.log(opacities / (1 - opacities)).unsqueeze(-1)
                        params['logit_opacities'].data.copy_(new_logit)

            # Original opacity-based removal
            to_remove = (torch.sigmoid(params['logit_opacities']) < remove_threshold).squeeze()
            # Original scale-based removal
            if iter >= prune_dict['remove_big_after']:
                big_points_ws = torch.exp(params['log_scales']).max(dim=1).values > 0.1 * variables['scene_radius']
                to_remove = torch.logical_or(to_remove, big_points_ws)
            params, variables = remove_points(to_remove, params, variables, optimizer)
            torch.cuda.empty_cache()

        # Reset Opacities
        if iter > 0 and iter % prune_dict['reset_opacities_every'] == 0 and prune_dict['reset_opacities']:
            new_params = {'logit_opacities': inverse_sigmoid(torch.ones_like(params['logit_opacities']) * 0.01)}
            params = update_params_and_optimizer(new_params, params, optimizer)

    return params, variables


# ============================================================
# INNOVATION 1+3: Three-way classifier (static/deforming/floater)
# ============================================================

def update_three_way_classifier(variables, gauss_vis, innovation_config):
    """
    Update visibility history buffer and classify Gaussians into:
    STATIC / DEFORMING / FLOATER based on visibility patterns.

    Args:
        variables: dict with vis_history, vis_frame_count, etc.
        gauss_vis: [N] per-Gaussian visibility from modified CUDA rasterizer
        innovation_config: dict with threshold settings
    """
    N = gauss_vis.shape[0]
    W = innovation_config.get('vis_window_size', 15)

    # Initialize circular buffer if needed
    if 'vis_history' not in variables:
        variables['vis_history'] = torch.zeros(N, W, device='cuda')
        variables['vis_frame_count'] = torch.zeros(N, device='cuda')
        variables['vis_write_idx'] = 0

    # Resize to match current Gaussian count (may have grown or shrunk)
    variables['vis_history'] = _resize_to_N(variables['vis_history'], N)
    variables['vis_frame_count'] = _resize_1d(variables['vis_frame_count'], N)

    # Write current visibility into circular buffer
    col = variables['vis_write_idx'] % W
    variables['vis_history'][:N, col] = gauss_vis[:N].detach()
    variables['vis_write_idx'] = (variables['vis_write_idx'] + 1) % W
    variables['vis_frame_count'][:N] += 1

    # === Three-way classification ===
    if innovation_config.get('enable_deformation', False):
        vis_threshold = innovation_config.get('vis_threshold', 0.3)
        var_threshold = innovation_config.get('var_threshold', 0.1)
        min_obs = innovation_config.get('min_observations', 5)

        vis_mean = variables['vis_history'][:N].mean(dim=1)
        vis_var = variables['vis_history'][:N].var(dim=1)
        enough_obs = variables['vis_frame_count'][:N] > min_obs

        # Deforming: visible but inconsistent across frames
        deform_mask = (vis_var > var_threshold) & (vis_mean > vis_threshold) & enough_obs
        variables['deform_mask'] = deform_mask

    return variables


# ============================================================
# Original densify function (unchanged)
# ============================================================

def densify(params, variables, optimizer, iter, densify_dict):
    if iter <= densify_dict['stop_after']:
        variables = accumulate_mean2d_gradient(variables)
        grad_thresh = densify_dict['grad_thresh']
        if (iter >= densify_dict['start_after']) and (iter % densify_dict['densify_every'] == 0):
            grads = variables['means2D_gradient_accum'] / variables['denom']
            grads[grads.isnan()] = 0.0
            to_clone = torch.logical_and(grads >= grad_thresh, (
                        torch.max(torch.exp(params['log_scales']), dim=1).values <= 0.01 * variables['scene_radius']))
            new_params = {k: v[to_clone] for k, v in params.items() if k not in ['cam_unnorm_rots', 'cam_trans']}
            params = cat_params_to_optimizer(new_params, params, optimizer)
            num_pts = params['means3D'].shape[0]

            padded_grad = torch.zeros(num_pts, device="cuda")
            padded_grad[:grads.shape[0]] = grads
            to_split = torch.logical_and(padded_grad >= grad_thresh,
                                         torch.max(torch.exp(params['log_scales']), dim=1).values > 0.01 * variables[
                                             'scene_radius'])
            n = densify_dict['num_to_split_into']
            new_params = {k: v[to_split].repeat(n, 1) for k, v in params.items() if k not in ['cam_unnorm_rots', 'cam_trans']}
            stds = torch.exp(params['log_scales'])[to_split].repeat(n, 3)
            means = torch.zeros((stds.size(0), 3), device="cuda")
            samples = torch.normal(mean=means, std=stds)
            rots = build_rotation(params['unnorm_rotations'][to_split]).repeat(n, 1, 1)
            new_params['means3D'] += torch.bmm(rots, samples.unsqueeze(-1)).squeeze(-1)
            new_params['log_scales'] = torch.log(torch.exp(new_params['log_scales']) / (0.8 * n))
            params = cat_params_to_optimizer(new_params, params, optimizer)
            num_pts = params['means3D'].shape[0]

            variables['means2D_gradient_accum'] = torch.zeros(num_pts, device="cuda")
            variables['denom'] = torch.zeros(num_pts, device="cuda")
            variables['max_2D_radius'] = torch.zeros(num_pts, device="cuda")
            to_remove = torch.cat((to_split, torch.zeros(n * to_split.sum(), dtype=torch.bool, device="cuda")))
            params, variables = remove_points(to_remove, params, variables, optimizer)

            if iter == densify_dict['stop_after']:
                remove_threshold = densify_dict['final_removal_opacity_threshold']
            else:
                remove_threshold = densify_dict['removal_opacity_threshold']
            to_remove = (torch.sigmoid(params['logit_opacities']) < remove_threshold).squeeze()
            if iter >= densify_dict['remove_big_after']:
                big_points_ws = torch.exp(params['log_scales']).max(dim=1).values > 0.1 * variables['scene_radius']
                to_remove = torch.logical_or(to_remove, big_points_ws)
            params, variables = remove_points(to_remove, params, variables, optimizer)

            torch.cuda.empty_cache()

        if iter > 0 and iter % densify_dict['reset_opacities_every'] == 0 and densify_dict['reset_opacities']:
            new_params = {'logit_opacities': inverse_sigmoid(torch.ones_like(params['logit_opacities']) * 0.01)}
            params = update_params_and_optimizer(new_params, params, optimizer)

    return params, variables


def update_learning_rate(optimizer, means3D_scheduler, iteration):
        for param_group in optimizer.param_groups:
            if param_group["name"] == "means3D":
                lr = means3D_scheduler(iteration)
                param_group['lr'] = lr
                return lr


def get_expon_lr_func(
    lr_init, lr_final, lr_delay_steps=0, lr_delay_mult=1.0, max_steps=1000000
):
    def helper(step):
        if step < 0 or (lr_init == 0.0 and lr_final == 0.0):
            return 0.0
        if lr_delay_steps > 0:
            delay_rate = lr_delay_mult + (1 - lr_delay_mult) * np.sin(
                0.5 * np.pi * np.clip(step / lr_delay_steps, 0, 1)
            )
        else:
            delay_rate = 1.0
        t = np.clip(step / max_steps, 0, 1)
        log_lerp = np.exp(np.log(lr_init) * (1 - t) + np.log(lr_final) * t)
        return delay_rate * log_lerp
    return helper

In [ ]:
%%writefile utils/slam_helpers.py
import torch
import torch.nn.functional as F
from utils.slam_external import build_rotation


def l1_loss_v1(x, y):
    return torch.abs((x - y)).mean()


def l1_loss_v2(x, y):
    return (torch.abs(x - y).sum(-1)).mean()


def weighted_l2_loss_v1(x, y, w):
    return torch.sqrt(((x - y) ** 2) * w + 1e-20).mean()


def weighted_l2_loss_v2(x, y, w):
    return torch.sqrt(((x - y) ** 2).sum(-1) * w + 1e-20).mean()


def quat_mult(q1, q2):
    w1, x1, y1, z1 = q1.T
    w2, x2, y2, z2 = q2.T
    w = w1 * w2 - x1 * x2 - y1 * y2 - z1 * z2
    x = w1 * x2 + x1 * w2 + y1 * z2 - z1 * y2
    y = w1 * y2 - x1 * z2 + y1 * w2 + z1 * x2
    z = w1 * z2 + x1 * y2 - y1 * x2 + z1 * w2
    return torch.stack([w, x, y, z]).T


def _sqrt_positive_part(x: torch.Tensor) -> torch.Tensor:
    """
    Returns torch.sqrt(torch.max(0, x))
    but with a zero subgradient where x is 0.
    Source: https://pytorch3d.readthedocs.io/en/latest/_modules/pytorch3d/transforms/rotation_conversions.html#matrix_to_quaternion
    """
    ret = torch.zeros_like(x)
    positive_mask = x > 0
    ret[positive_mask] = torch.sqrt(x[positive_mask])
    return ret


def matrix_to_quaternion(matrix: torch.Tensor) -> torch.Tensor:
    """
    Convert rotations given as rotation matrices to quaternions.

    Args:
        matrix: Rotation matrices as tensor of shape (..., 3, 3).

    Returns:
        quaternions with real part first, as tensor of shape (..., 4).
    Source: https://pytorch3d.readthedocs.io/en/latest/_modules/pytorch3d/transforms/rotation_conversions.html#matrix_to_quaternion
    """
    if matrix.size(-1) != 3 or matrix.size(-2) != 3:
        raise ValueError(f"Invalid rotation matrix shape {matrix.shape}.")

    batch_dim = matrix.shape[:-2]
    m00, m01, m02, m10, m11, m12, m20, m21, m22 = torch.unbind(
        matrix.reshape(batch_dim + (9,)), dim=-1
    )

    q_abs = _sqrt_positive_part(
        torch.stack(
            [
                1.0 + m00 + m11 + m22,
                1.0 + m00 - m11 - m22,
                1.0 - m00 + m11 - m22,
                1.0 - m00 - m11 + m22,
            ],
            dim=-1,
        )
    )

    quat_by_rijk = torch.stack(
        [
            torch.stack([q_abs[..., 0] ** 2, m21 - m12, m02 - m20, m10 - m01], dim=-1),
            torch.stack([m21 - m12, q_abs[..., 1] ** 2, m10 + m01, m02 + m20], dim=-1),
            torch.stack([m02 - m20, m10 + m01, q_abs[..., 2] ** 2, m12 + m21], dim=-1),
            torch.stack([m10 - m01, m20 + m02, m21 + m12, q_abs[..., 3] ** 2], dim=-1),
        ],
        dim=-2,
    )

    flr = torch.tensor(0.1).to(dtype=q_abs.dtype, device=q_abs.device)
    quat_candidates = quat_by_rijk / (2.0 * q_abs[..., None].max(flr))

    return quat_candidates[
        F.one_hot(q_abs.argmax(dim=-1), num_classes=4) > 0.5, :
    ].reshape(batch_dim + (4,))


def transformed_params2rendervar(params, transformed_pts):
    rendervar = {
        'means3D': transformed_pts,
        'rotations': F.normalize(params['unnorm_rotations']),
        'opacities': torch.sigmoid(params['logit_opacities']),
        'means2D': torch.zeros_like(params['means3D'], requires_grad=True, device="cuda") + 0
    }
    if params['log_scales'].shape[1] == 1:
        rendervar['colors_precomp'] = params['rgb_colors']
        rendervar['scales'] = torch.exp(torch.tile(params['log_scales'], (1, 3)))
    else:
        rendervar['shs'] = torch.cat((params['rgb_colors'].reshape(params['rgb_colors'].shape[0], 3, -1).transpose(1, 2), params['feature_rest'].reshape(params['rgb_colors'].shape[0], 3, -1).transpose(1, 2)), dim=1)
        rendervar['scales'] = torch.exp(params['log_scales'])
    return rendervar


def get_depth_and_silhouette(pts_3D, w2c):
    """
    Function to compute depth and silhouette for each gaussian.
    These are evaluated at gaussian center.
    """
    # Depth of each gaussian center in camera frame
    pts4 = torch.cat((pts_3D, torch.ones_like(pts_3D[:, :1])), dim=-1)
    pts_in_cam = (w2c @ pts4.transpose(0, 1)).transpose(0, 1)
    depth_z = pts_in_cam[:, 2].unsqueeze(-1) # [num_gaussians, 1]
    depth_z_sq = torch.square(depth_z) # [num_gaussians, 1]

    # Depth and Silhouette
    depth_silhouette = torch.zeros((pts_3D.shape[0], 3)).cuda().float()
    depth_silhouette[:, 0] = depth_z.squeeze(-1)
    depth_silhouette[:, 1] = 1.0
    depth_silhouette[:, 2] = depth_z_sq.squeeze(-1)
    
    return depth_silhouette


def transformed_params2depthplussilhouette(params, w2c, transformed_pts):
    rendervar = {
        'means3D': transformed_pts,
        'colors_precomp': get_depth_and_silhouette(transformed_pts, w2c),
        'rotations': F.normalize(params['unnorm_rotations']),
        'opacities': torch.sigmoid(params['logit_opacities']),
        'means2D': torch.zeros_like(params['means3D'], requires_grad=True, device="cuda") + 0
    }
    if params['log_scales'].shape[1] == 1:
        rendervar['scales'] = torch.exp(torch.tile(params['log_scales'], (1, 3)))
    else:
        rendervar['scales'] = torch.exp(params['log_scales'])
    return rendervar


# ============================================================
# MODIFIED: Innovation 3 - Deformation-aware transform_to_frame
# ============================================================

def transform_to_frame(params, time_idx, gaussians_grad, camera_grad,
                       apply_deformation=False, variables=None):
    """
    Function to transform Isotropic Gaussians from world frame to camera frame.
    
    Innovation 3: Optionally applies per-Gaussian deformation offsets for
    Gaussians classified as DEFORMING by the three-way classifier.
    
    Args:
        params: dict of parameters
        time_idx: time index to transform to
        gaussians_grad: enable gradients for Gaussians
        camera_grad: enable gradients for camera pose
        apply_deformation: whether to apply deformation offsets (Innovation 3)
        variables: dict with deform_mask etc. (needed when apply_deformation=True)
    
    Returns:
        transformed_pts: Transformed Centers of Gaussians
    """
    # Get Frame Camera Pose
    if camera_grad:
        cam_rot = F.normalize(params['cam_unnorm_rots'][..., time_idx])
        cam_tran = params['cam_trans'][..., time_idx]
    else:
        cam_rot = F.normalize(params['cam_unnorm_rots'][..., time_idx].detach())
        cam_tran = params['cam_trans'][..., time_idx].detach()
    rel_w2c = torch.eye(4).cuda().float()
    rel_w2c[:3, :3] = build_rotation(cam_rot)
    rel_w2c[:3, 3] = cam_tran

    # Get Centers and norm Rots of Gaussians in World Frame
    if gaussians_grad:
        pts = params['means3D']
    else:
        pts = params['means3D'].detach()
    
    # === INNOVATION 3: Apply deformation offsets ===
    if (apply_deformation and 'deform_offsets' in params and 
        variables is not None and 'deform_mask' in variables):
        deform_mask = variables['deform_mask']
        if deform_mask.shape[0] == pts.shape[0] and deform_mask.any():
            pts = pts.clone()  # avoid in-place modification
            pts[deform_mask] = pts[deform_mask] + params['deform_offsets'][deform_mask]
    
    # Transform Centers and Unnorm Rots of Gaussians to Camera Frame
    pts_ones = torch.ones(pts.shape[0], 1).cuda().float()
    pts4 = torch.cat((pts, pts_ones), dim=1)
    transformed_pts = (rel_w2c @ pts4.T).T[:, :3]

    return transformed_pts


def transform_to_frame_eval(params, camrt=None, rel_w2c=None):
    """
    Function to transform Isotropic Gaussians from world frame to camera frame.
    Used for evaluation only (no deformation applied).
    """
    # Get Frame Camera Pose
    if rel_w2c is None:
        cam_rot, cam_tran = camrt
        rel_w2c = torch.eye(4).cuda().float()
        rel_w2c[:3, :3] = build_rotation(cam_rot)
        rel_w2c[:3, 3] = cam_tran

    # Get Centers and norm Rots of Gaussians in World Frame
    pts = params['means3D'].detach()
    
    # Transform Centers and Unnorm Rots of Gaussians to Camera Frame
    pts_ones = torch.ones(pts.shape[0], 1).cuda().float()
    pts4 = torch.cat((pts, pts_ones), dim=1)
    transformed_pts = (rel_w2c @ pts4.T).T[:, :3]

    return transformed_pts

In [ ]:
%%writefile scripts/main.py
import argparse
import os
import shutil
import sys
import time
from importlib.machinery import SourceFileLoader

_BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))

sys.path.insert(0, _BASE_DIR)

print("System Paths:")
for p in sys.path:
    print(p)

import numpy as np
import torch
import torch.nn.functional as F
from tqdm import tqdm

from datasets.gradslam_datasets import (
    load_dataset_config,
    EndoSLAMDataset,
    C3VDDataset
)
from utils.common_utils import seed_everything, save_params_ckpt, save_params, save_means3D
from utils.eval_helpers import report_progress, eval_save
from utils.keyframe_selection import keyframe_selection_overlap, keyframe_selection_distance
from utils.recon_helpers import setup_camera, energy_mask
from utils.slam_helpers import (
    transformed_params2rendervar, transformed_params2depthplussilhouette,
    transform_to_frame, l1_loss_v1, matrix_to_quaternion
)
from utils.slam_external import calc_ssim, build_rotation, prune_gaussians, densify, update_three_way_classifier
from utils.vis_utils import plot_video
from utils.time_helper import Timer

from diff_gaussian_rasterization import GaussianRasterizer as Renderer



def get_dataset(config_dict, basedir, sequence, **kwargs):
    if config_dict["dataset_name"].lower() in ["endoslam_unity"]:
        return EndoSLAMDataset(config_dict, basedir, sequence, **kwargs)
    elif config_dict["dataset_name"].lower() in ["c3vd"]:
        return C3VDDataset(config_dict, basedir, sequence, **kwargs)
    else:
        raise ValueError(f"Unknown dataset name {config_dict['dataset_name']}")


def get_pointcloud(color, depth, intrinsics, w2c, transform_pts=True, 
                   mask=None, compute_mean_sq_dist=False, mean_sq_dist_method="projective"):
    width, height = color.shape[2], color.shape[1]
    CX = intrinsics[0][2]
    CY = intrinsics[1][2]
    FX = intrinsics[0][0]
    FY = intrinsics[1][1]

    # Compute indices of pixels
    x_grid, y_grid = torch.meshgrid(torch.arange(width).cuda().float(),
                                    torch.arange(height).cuda().float(),
                                    indexing='xy')
    xx = (x_grid - CX)/FX
    yy = (y_grid - CY)/FY
    xx = xx.reshape(-1)
    yy = yy.reshape(-1)
    depth_z = depth[0].reshape(-1)

    # Initialize point cloud
    pts_cam = torch.stack((xx * depth_z, yy * depth_z, depth_z), dim=-1)
    if transform_pts:
        pix_ones = torch.ones(height * width, 1).cuda().float()
        pts4 = torch.cat((pts_cam, pix_ones), dim=1)
        c2w = torch.inverse(w2c)
        pts = (c2w @ pts4.T).T[:, :3]
    else:
        pts = pts_cam

    # Compute mean squared distance for initializing the scale of the Gaussians
    if compute_mean_sq_dist:
        if mean_sq_dist_method == "projective":
            # Projective Geometry (this is fast, farther -> larger radius)
            scale_gaussian = depth_z / ((FX + FY)/2)
            mean3_sq_dist = scale_gaussian**2
        else:
            raise ValueError(f"Unknown mean_sq_dist_method {mean_sq_dist_method}")
    
    # Colorize point cloud
    cols = torch.permute(color, (1, 2, 0)).reshape(-1, 3) # (C, H, W) -> (H, W, C) -> (H * W, C)
    point_cld = torch.cat((pts, cols), -1)
    # Image.fromarray(np.uint8((torch.permute(color, (1, 2, 0)) * mask.reshape(320, 320, 1)).detach().cpu().numpy()*255), 'RGB').save('gaussian.png')

    # Select points based on mask
    if mask is not None:
        point_cld = point_cld[mask]
        if compute_mean_sq_dist:
            mean3_sq_dist = mean3_sq_dist[mask]

    if compute_mean_sq_dist:
        return point_cld, mean3_sq_dist
    else:
        return point_cld


def initialize_params(init_pt_cld, num_frames, mean3_sq_dist, use_simplification=True):
    num_pts = init_pt_cld.shape[0]
    means3D = init_pt_cld[:, :3] # [num_gaussians, 3]
    unnorm_rots = np.tile([1, 0, 0, 0], (num_pts, 1)) # [num_gaussians, 3]
    logit_opacities = torch.zeros((num_pts, 1), dtype=torch.float, device="cuda")
    params = {
        'means3D': means3D,
        'rgb_colors': init_pt_cld[:, 3:6],
        'unnorm_rotations': unnorm_rots,
        'logit_opacities': logit_opacities,
        'log_scales': torch.tile(torch.log(torch.sqrt(mean3_sq_dist))[..., None], (1, 1 if use_simplification else 3)),
    }
    if not use_simplification:
        params['feature_rest'] = torch.zeros(num_pts, 45) # set SH degree 3 fixed

    # Initialize a single gaussian trajectory to model the camera poses relative to the first frame
    cam_rots = np.tile([1, 0, 0, 0], (1, 1))
    cam_rots = np.tile(cam_rots[:, :, None], (1, 1, num_frames))
    params['cam_unnorm_rots'] = cam_rots
    params['cam_trans'] = np.zeros((1, 3, num_frames))

    for k, v in params.items():
        # Check if value is already a torch tensor
        if not isinstance(v, torch.Tensor):
            params[k] = torch.nn.Parameter(torch.tensor(v).cuda().float().contiguous().requires_grad_(True))
        else:
            params[k] = torch.nn.Parameter(v.cuda().float().contiguous().requires_grad_(True))

    variables = {'max_2D_radius': torch.zeros(params['means3D'].shape[0]).cuda().float(),
                 'means2D_gradient_accum': torch.zeros(params['means3D'].shape[0]).cuda().float(),
                 'denom': torch.zeros(params['means3D'].shape[0]).cuda().float(),
                 'timestep': torch.zeros(params['means3D'].shape[0]).cuda().float(),
                 'deform_mask': torch.zeros(params['means3D'].shape[0], dtype=torch.bool, device='cuda')}

    return params, variables


def initialize_optimizer(params, lrs_dict):
    lrs = lrs_dict
    param_groups = [{'params': [v], 'name': k, 'lr': lrs.get(k, 0.0)} for k, v in params.items() if k != 'feature_rest']
    if 'feature_rest' in params:
        param_groups.append({'params': [params['feature_rest']], 'name': 'feature_rest', 'lr': lrs.get('rgb_colors', 0.0) / 20.0})
    return torch.optim.Adam(param_groups, lr=0.0, eps=1e-15)


def initialize_first_timestep(dataset, num_frames, scene_radius_depth_ratio, mean_sq_dist_method, densify_dataset=None, use_simplification=True):
    # Get RGB-D Data & Camera Parameters
    color, depth, intrinsics, pose = dataset[0]

    # Process RGB-D Data
    color = color.permute(2, 0, 1) / 255 # (H, W, C) -> (C, H, W)
    depth = depth.permute(2, 0, 1) # (H, W, C) -> (C, H, W)
    
    # Process Camera Parameters
    intrinsics = intrinsics[:3, :3]
    w2c = torch.linalg.inv(pose)

    # Setup Camera
    cam = setup_camera(color.shape[2], color.shape[1], intrinsics.cpu().numpy(), w2c.detach().cpu().numpy(), use_simplification=use_simplification)

    if densify_dataset is not None:
        # Get Densification RGB-D Data & Camera Parameters
        color, depth, densify_intrinsics, _ = densify_dataset[0]
        color = color.permute(2, 0, 1) / 255 # (H, W, C) -> (C, H, W)
        depth = depth.permute(2, 0, 1) # (H, W, C) -> (C, H, W)
        densify_intrinsics = densify_intrinsics[:3, :3]
        densify_cam = setup_camera(color.shape[2], color.shape[1], densify_intrinsics.cpu().numpy(), w2c.detach().cpu().numpy())
    else:
        densify_intrinsics = intrinsics

    # Get Initial Point Cloud (PyTorch CUDA Tensor)

    mask = (depth > 0) & energy_mask(color) # Mask out invalid depth values
    # Image.fromarray(np.uint8(mask[0].detach().cpu().numpy()*255), 'L').save('mask.png')
    mask = mask.reshape(-1)
    init_pt_cld, mean3_sq_dist = get_pointcloud(color, depth, densify_intrinsics, w2c, 
                                                mask=mask, compute_mean_sq_dist=True, 
                                                mean_sq_dist_method=mean_sq_dist_method)

    # Initialize Parameters
    params, variables = initialize_params(init_pt_cld, num_frames, mean3_sq_dist, use_simplification)

    # Initialize an estimate of scene radius for Gaussian-Splatting Densification
    variables['scene_radius'] = torch.max(depth)/scene_radius_depth_ratio # NOTE: change_here

    # Innovation 3: Initialize deformation offsets
    params['deform_offsets'] = torch.nn.Parameter(
        torch.zeros(params['means3D'].shape[0], 3, device='cuda').requires_grad_(True))

    if densify_dataset is not None:
        return params, variables, intrinsics, w2c, cam, densify_intrinsics, densify_cam
    else:
        return params, variables, intrinsics, w2c, cam


def get_loss(params, curr_data, variables, iter_time_idx, loss_weights, use_sil_for_loss, 
             sil_thres, use_l1,ignore_outlier_depth_loss, tracking=False, 
             mapping=False, do_ba=False, plot_dir=None, visualize_tracking_loss=False, tracking_iteration=None):
    global w2cs, w2ci
    # Initialize Loss Dictionary
    losses = {}

    if tracking:
        # Get current frame Gaussians, where only the camera pose gets gradient
        transformed_pts = transform_to_frame(params, iter_time_idx, 
                                             gaussians_grad=False,
                                             camera_grad=True)
    elif mapping:
        if do_ba: # Bundle Adjustment
            # Get current frame Gaussians, where both camera pose and Gaussians get gradient
            transformed_pts = transform_to_frame(params, iter_time_idx,
                                                 gaussians_grad=True,
                                                 camera_grad=True,
                                                 apply_deformation=True, variables=variables)
        else:
            # Get current frame Gaussians, where only the Gaussians get gradient
            transformed_pts = transform_to_frame(params, iter_time_idx,
                                                 gaussians_grad=True,
                                                 camera_grad=False,
                                                 apply_deformation=True, variables=variables)
    else:
        # Get current frame Gaussians, where only the Gaussians get gradient
        transformed_pts = transform_to_frame(params, iter_time_idx,
                                             gaussians_grad=True,
                                             camera_grad=False,
                                             apply_deformation=True, variables=variables)

    # Initialize Render Variables
    rendervar = transformed_params2rendervar(params, transformed_pts)
    depth_sil_rendervar = transformed_params2depthplussilhouette(params, curr_data['w2c'],
                                                                 transformed_pts)
    
    # Visualize the Rendered Images
    # online_render(curr_data, iter_time_idx, rendervar, dev_use_controller=False)
        
    # RGB Rendering
    rendervar['means2D'].retain_grad()
    im, radius, _, gauss_vis = Renderer(raster_settings=curr_data['cam'])(**rendervar)
    variables['means2D'] = rendervar['means2D'] # Gradient only accum from colour render for densification


    # Depth & Silhouette Rendering
    depth_sil, _, _, _ = Renderer(raster_settings=curr_data['cam'])(**depth_sil_rendervar)
    depth = depth_sil[0, :, :].unsqueeze(0)
    silhouette = depth_sil[1, :, :]
    presence_sil_mask = (silhouette > sil_thres)
    depth_sq = depth_sil[2, :, :].unsqueeze(0)
    uncertainty = depth_sq - depth**2
    uncertainty = uncertainty.detach()

    # Mask with valid depth values (accounts for outlier depth values)
    nan_mask = (~torch.isnan(depth)) & (~torch.isnan(uncertainty))
    bg_mask = energy_mask(curr_data['im'])
    if ignore_outlier_depth_loss:
        depth_error = torch.abs(curr_data['depth'] - depth) * (curr_data['depth'] > 0)
        mask = (depth_error < 20*depth_error.mean())
        mask = mask & (curr_data['depth'] > 0)
    else:
        mask = (curr_data['depth'] > 0)
    mask = mask & nan_mask & bg_mask
    # Mask with presence silhouette mask (accounts for empty space)
    if tracking and use_sil_for_loss:
        mask = mask & presence_sil_mask

    # Depth loss
    if use_l1:
        mask = mask.detach()
        if tracking:
            losses['depth'] = torch.abs(curr_data['depth'] - depth)[mask].sum()
        else:
            losses['depth'] = torch.abs(curr_data['depth'] - depth)[mask].mean()
    # RGB Loss
    if tracking and (use_sil_for_loss or ignore_outlier_depth_loss):
        color_mask = torch.tile(mask, (3, 1, 1))
        color_mask = color_mask.detach()
        losses['im'] = torch.abs(curr_data['im'] - im)[color_mask].sum()
    elif tracking:
        losses['im'] = torch.abs(curr_data['im'] - im).sum()
    else:
        losses['im'] = 0.8 * l1_loss_v1(im, curr_data['im']) + 0.2 * (1.0 - calc_ssim(im, curr_data['im']))

    weighted_losses = {k: v * loss_weights[k] for k, v in losses.items()}
    loss = sum(weighted_losses.values())

    # === Innovation 3: Deformation regularization losses ===
    if mapping and 'deform_offsets' in params:
        # Magnitude regularization: penalize large deformation offsets
        deform_mag = torch.norm(params['deform_offsets'], dim=1).mean()
        lambda_mag = 0.01  # default weight for magnitude regularization
        loss = loss + lambda_mag * deform_mag

        # Temporal smoothness regularization: penalize change from previous offsets
        if 'prev_deform_offsets' in variables:
            prev_offsets = variables['prev_deform_offsets']
            if prev_offsets.shape[0] == params['deform_offsets'].shape[0]:
                temporal_diff = torch.norm(params['deform_offsets'] - prev_offsets, dim=1).mean()
                lambda_temp = 0.005  # default weight for temporal regularization
                loss = loss + lambda_temp * temporal_diff

    seen = radius > 0
    variables['max_2D_radius'][seen] = torch.max(radius[seen], variables['max_2D_radius'][seen])
    variables['seen'] = seen
    variables['gauss_vis'] = gauss_vis.detach()
    weighted_losses['loss'] = loss

    return loss, variables, weighted_losses


def initialize_new_params(new_pt_cld, mean3_sq_dist, use_simplification):
    num_pts = new_pt_cld.shape[0]
    means3D = new_pt_cld[:, :3] # [num_gaussians, 3]
    unnorm_rots = np.tile([1, 0, 0, 0], (num_pts, 1)) # [num_gaussians, 3]
    logit_opacities = torch.ones((num_pts, 1), dtype=torch.float, device="cuda") * 0.5
    params = {
        'means3D': means3D,
        'rgb_colors': new_pt_cld[:, 3:6],
        'unnorm_rotations': unnorm_rots,
        'logit_opacities': logit_opacities,
        'log_scales': torch.tile(torch.log(torch.sqrt(mean3_sq_dist))[..., None], (1, 1 if use_simplification else 3)),
    }
    if not use_simplification:
        params['feature_rest'] = torch.zeros(num_pts, 45) # set SH degree 3 fixed
    for k, v in params.items():
        # Check if value is already a torch tensor
        if not isinstance(v, torch.Tensor):
            params[k] = torch.nn.Parameter(torch.tensor(v).cuda().float().contiguous().requires_grad_(True))
        else:
            params[k] = torch.nn.Parameter(v.cuda().float().contiguous().requires_grad_(True))

    return params


def add_new_gaussians(params, variables, curr_data, sil_thres, time_idx, mean_sq_dist_method, use_simplification=True):
    # Silhouette Rendering
    transformed_pts = transform_to_frame(params, time_idx, gaussians_grad=False, camera_grad=False)
    depth_sil_rendervar = transformed_params2depthplussilhouette(params, curr_data['w2c'],
                                                                 transformed_pts)
    depth_sil, _, _, _ = Renderer(raster_settings=curr_data['cam'])(**depth_sil_rendervar)
    silhouette = depth_sil[1, :, :]
    non_presence_sil_mask = (silhouette < sil_thres)
    # Check for new foreground objects by using GT depth
    gt_depth = curr_data['depth'][0, :, :]
    render_depth = depth_sil[0, :, :]
    depth_error = torch.abs(gt_depth - render_depth) * (gt_depth > 0)
    non_presence_depth_mask = (render_depth > gt_depth) * (depth_error > 20*depth_error.mean())
    # Determine non-presence mask
    non_presence_mask = non_presence_sil_mask | non_presence_depth_mask
    # Flatten mask
    non_presence_mask = non_presence_mask.reshape(-1)

    # Get the new frame Gaussians based on the Silhouette
    if torch.sum(non_presence_mask) > 0:
        # Get the new pointcloud in the world frame
        curr_cam_rot = torch.nn.functional.normalize(params['cam_unnorm_rots'][..., time_idx].detach())
        curr_cam_tran = params['cam_trans'][..., time_idx].detach()
        curr_w2c = torch.eye(4).cuda().float()
        curr_w2c[:3, :3] = build_rotation(curr_cam_rot)
        curr_w2c[:3, 3] = curr_cam_tran
        valid_depth_mask = (curr_data['depth'][0, :, :] > 0) & (curr_data['depth'][0, :, :] < 1e10)
        non_presence_mask = non_presence_mask & valid_depth_mask.reshape(-1)
        valid_color_mask = energy_mask(curr_data['im']).squeeze()
        non_presence_mask = non_presence_mask & valid_color_mask.reshape(-1)        
        new_pt_cld, mean3_sq_dist = get_pointcloud(curr_data['im'], curr_data['depth'], curr_data['intrinsics'], 
                                    curr_w2c, mask=non_presence_mask, compute_mean_sq_dist=True,
                                    mean_sq_dist_method=mean_sq_dist_method)
        new_params = initialize_new_params(new_pt_cld, mean3_sq_dist, use_simplification)
        for k, v in new_params.items():
            params[k] = torch.nn.Parameter(torch.cat((params[k], v), dim=0).requires_grad_(True))
        num_pts = params['means3D'].shape[0]
        variables['means2D_gradient_accum'] = torch.zeros(num_pts, device="cuda").float()
        variables['denom'] = torch.zeros(num_pts, device="cuda").float()
        variables['max_2D_radius'] = torch.zeros(num_pts, device="cuda").float()
        new_timestep = time_idx*torch.ones(new_pt_cld.shape[0],device="cuda").float()
        variables['timestep'] = torch.cat((variables['timestep'], new_timestep),dim=0)

        # Innovation 3: Add deformation offsets for new Gaussians
        if 'deform_offsets' in params:
            new_deform = torch.zeros(new_pt_cld.shape[0], 3, device='cuda')
            params['deform_offsets'] = torch.nn.Parameter(
                torch.cat((params['deform_offsets'], new_deform), dim=0).requires_grad_(True))

    return params, variables


def initialize_camera_pose(params, curr_time_idx, forward_prop):
    with torch.no_grad():
        if curr_time_idx > 1 and forward_prop:
            # Initialize the camera pose for the current frame based on a constant velocity model
            # Rotation
            prev_rot1 = F.normalize(params['cam_unnorm_rots'][..., curr_time_idx-1].detach())
            prev_rot2 = F.normalize(params['cam_unnorm_rots'][..., curr_time_idx-2].detach())
            new_rot = F.normalize(prev_rot1 + (prev_rot1 - prev_rot2))
            params['cam_unnorm_rots'][..., curr_time_idx] = new_rot.detach()
            # Translation
            prev_tran1 = params['cam_trans'][..., curr_time_idx-1].detach()
            prev_tran2 = params['cam_trans'][..., curr_time_idx-2].detach()
            new_tran = prev_tran1 + (prev_tran1 - prev_tran2)
            params['cam_trans'][..., curr_time_idx] = new_tran.detach()
        else:
            # Initialize the camera pose for the current frame
            params['cam_unnorm_rots'][..., curr_time_idx] = params['cam_unnorm_rots'][..., curr_time_idx-1].detach()
            params['cam_trans'][..., curr_time_idx] = params['cam_trans'][..., curr_time_idx-1].detach()
    
    return params


def convert_params_to_store(params):
    params_to_store = {}
    for k, v in params.items():
        if isinstance(v, torch.Tensor):
            params_to_store[k] = v.detach().clone()
        else:
            params_to_store[k] = v
    return params_to_store


# ============================================================
# Innovation 2: Bundle Adjustment helper functions
# ============================================================

def select_ba_keyframes(keyframe_list, n_keyframes, strategy='uniform'):
    """
    Select keyframes for periodic bundle adjustment.
    
    Args:
        keyframe_list: list of keyframe dicts
        n_keyframes: number of keyframes to select
        strategy: 'uniform' for uniformly spaced, 'recent' for most recent
    
    Returns:
        selected_indices: list of indices into keyframe_list
    """
    n_available = len(keyframe_list)
    if n_available <= n_keyframes:
        return list(range(n_available))
    
    if strategy == 'uniform':
        # Uniformly space keyframes, always include first and last
        indices = np.linspace(0, n_available - 1, n_keyframes, dtype=int).tolist()
        return sorted(set(indices))
    elif strategy == 'recent':
        # Select most recent keyframes
        return list(range(n_available - n_keyframes, n_available))
    else:
        # Default: uniform
        indices = np.linspace(0, n_available - 1, n_keyframes, dtype=int).tolist()
        return sorted(set(indices))


def periodic_bundle_adjustment(params, variables, keyframe_list, config, intrinsics, cam, first_frame_w2c):
    """
    Perform periodic bundle adjustment over selected keyframes.
    
    Innovation 2: Jointly optimizes camera poses and Gaussian parameters
    over a subset of keyframes to reduce drift.
    
    Args:
        params: current parameters
        variables: current variables
        keyframe_list: list of keyframe dicts
        config: full config dict
        intrinsics: camera intrinsics
        cam: camera setup object
        first_frame_w2c: first frame world-to-camera transform
    
    Returns:
        params, variables: updated parameters and variables
    """
    innovation_cfg = config.get('innovations', {})
    ba_num_iters = innovation_cfg.get('ba_num_iters', 50)
    ba_n_keyframes = innovation_cfg.get('ba_n_keyframes', 10)
    ba_keyframe_strategy = innovation_cfg.get('ba_keyframe_strategy', 'uniform')
    
    # Select keyframes for BA
    selected_kf_indices = select_ba_keyframes(keyframe_list, ba_n_keyframes, ba_keyframe_strategy)
    
    if len(selected_kf_indices) < 2:
        return params, variables
    
    print(f"  BA: optimizing over {len(selected_kf_indices)} keyframes for {ba_num_iters} iterations")
    
    # Use mapping learning rates for BA
    optimizer = initialize_optimizer(params, config['mapping']['lrs'])
    
    for ba_iter in range(ba_num_iters):
        # Randomly pick a keyframe from the selected set
        rand_idx = np.random.randint(0, len(selected_kf_indices))
        kf_idx = selected_kf_indices[rand_idx]
        kf = keyframe_list[kf_idx]
        
        iter_time_idx = kf['id']
        iter_color = kf['color']
        iter_depth = kf['depth']
        
        iter_data = {
            'cam': cam, 'im': iter_color, 'depth': iter_depth, 'id': iter_time_idx,
            'intrinsics': intrinsics, 'w2c': first_frame_w2c, 'iter_gt_w2c_list': None
        }
        
        # Loss with BA (both camera and gaussians get gradient)
        loss, variables, losses = get_loss(
            params, iter_data, variables, iter_time_idx,
            config['mapping']['loss_weights'],
            config['mapping']['use_sil_for_loss'],
            config['mapping']['sil_thres'],
            config['mapping']['use_l1'],
            config['mapping']['ignore_outlier_depth_loss'],
            mapping=True, do_ba=True
        )
        
        loss.backward()
        with torch.no_grad():
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
    
    print(f"  BA: completed, final loss = {loss.item():.6f}")
    return params, variables


def rgbd_slam(config: dict):
    # timer = Timer()
    # timer.start()
    
    # Print Config
    print("Loaded Config:")
    if "use_depth_loss_thres" not in config['tracking']:
        config['tracking']['use_depth_loss_thres'] = False
        config['tracking']['depth_loss_thres'] = 100000
    if "visualize_tracking_loss" not in config['tracking']:
        config['tracking']['visualize_tracking_loss'] = False
    print(f"{config}")

    # Create Output Directories
    output_dir = os.path.join(config["workdir"], config["run_name"])
    eval_dir = os.path.join(output_dir, "eval")
    os.makedirs(eval_dir, exist_ok=True)

    # Get Device
    device = torch.device(config["primary_device"])

    # Load Dataset
    print("Loading Dataset ...")
    dataset_config = config["data"]
    if 'distance_keyframe_selection' not in config:
        config['distance_keyframe_selection'] = False
    if config['distance_keyframe_selection']:
        print("Using CDF Keyframe Selection. Note that \'mapping window size\' is useless.")
        if 'distance_current_frame_prob' not in config:
            config['distance_current_frame_prob'] = 0.5
    if 'gaussian_simplification' not in config:
        config['gaussian_simplification'] = True # simplified in paper
    if not config['gaussian_simplification']:
        print("Using Full Gaussian Representation, which may cause unstable optimization if not fully optimized.")
    if "gradslam_data_cfg" not in dataset_config:
        gradslam_data_cfg = {}
        gradslam_data_cfg["dataset_name"] = dataset_config["dataset_name"]
    else:
        gradslam_data_cfg = load_dataset_config(dataset_config["gradslam_data_cfg"])
    if "train_or_test" not in dataset_config:
        dataset_config["train_or_test"] = 'all'
    if "preload" not in dataset_config:
        dataset_config["preload"] = False
    if "ignore_bad" not in dataset_config:
        dataset_config["ignore_bad"] = False
    if "use_train_split" not in dataset_config:
        dataset_config["use_train_split"] = True
    if "densification_image_height" not in dataset_config:
        dataset_config["densification_image_height"] = dataset_config["desired_image_height"]
        dataset_config["densification_image_width"] = dataset_config["desired_image_width"]
        seperate_densification_res = False
    else:
        if dataset_config["densification_image_height"] != dataset_config["desired_image_height"] or \
            dataset_config["densification_image_width"] != dataset_config["desired_image_width"]:
            seperate_densification_res = True
        else:
            seperate_densification_res = False
    if "tracking_image_height" not in dataset_config:
        dataset_config["tracking_image_height"] = dataset_config["desired_image_height"]
        dataset_config["tracking_image_width"] = dataset_config["desired_image_width"]
        seperate_tracking_res = False
    else:
        if dataset_config["tracking_image_height"] != dataset_config["desired_image_height"] or \
            dataset_config["tracking_image_width"] != dataset_config["desired_image_width"]:
            seperate_tracking_res = True
        else:
            seperate_tracking_res = False
    # Poses are relative to the first frame
    dataset = get_dataset(
        config_dict=gradslam_data_cfg,
        basedir=dataset_config["basedir"],
        sequence=os.path.basename(dataset_config["sequence"]),
        start=dataset_config["start"],
        end=dataset_config["end"],
        stride=dataset_config["stride"],
        desired_height=dataset_config["desired_image_height"],
        desired_width=dataset_config["desired_image_width"],
        device=device,
        relative_pose=True,
        ignore_bad=dataset_config["ignore_bad"],
        use_train_split=dataset_config["use_train_split"],
        train_or_test=dataset_config["train_or_test"]
    )
    num_frames = dataset_config["num_frames"]
    if num_frames == -1:
        num_frames = len(dataset)

    if dataset_config["train_or_test"] == 'train': # kind of ill implementation here. train_or_test should be 'all' or 'train'. If 'test', you view test set as full dataset.
        eval_dataset = get_dataset(
            config_dict=gradslam_data_cfg,
            basedir=dataset_config["basedir"],
            sequence=os.path.basename(dataset_config["sequence"]),
            start=dataset_config["start"],
            end=dataset_config["end"],
            stride=dataset_config["stride"],
            desired_height=dataset_config["desired_image_height"], # if you eval, you should keep reso as raw image.
            desired_width=dataset_config["desired_image_width"],
            device=device,
            relative_pose=True,
            ignore_bad=dataset_config["ignore_bad"],
            use_train_split=dataset_config["use_train_split"],
            train_or_test='test'
        )
    # Init seperate dataloader for densification if required
    if seperate_densification_res:
        densify_dataset = get_dataset(
            config_dict=gradslam_data_cfg,
            basedir=dataset_config["basedir"],
            sequence=os.path.basename(dataset_config["sequence"]),
            start=dataset_config["start"],
            end=dataset_config["end"],
            stride=dataset_config["stride"],
            desired_height=dataset_config["densification_image_height"],
            desired_width=dataset_config["densification_image_width"],
            device=device,
            relative_pose=True,
            preload = dataset_config["preload"],
            ignore_bad=dataset_config["ignore_bad"],
            use_train_split=dataset_config["use_train_split"],
            train_or_test=dataset_config["train_or_test"]
        )
        # Initialize Parameters, Canonical & Densification Camera parameters
        params, variables, intrinsics, first_frame_w2c, cam, \
            densify_intrinsics, densify_cam = initialize_first_timestep(dataset, num_frames,
                                                                        config['scene_radius_depth_ratio'],
                                                                        config['mean_sq_dist_method'],
                                                                        densify_dataset=densify_dataset, 
                                                                        use_simplification=config['gaussian_simplification'])                                                                                                                  
    else:
        # Initialize Parameters & Canoncial Camera parameters
        params, variables, intrinsics, first_frame_w2c, cam = initialize_first_timestep(dataset, num_frames, 
                                                                                        config['scene_radius_depth_ratio'],
                                                                                        config['mean_sq_dist_method'], 
                                                                                        use_simplification=config['gaussian_simplification'])
    
    # Init seperate dataloader for tracking if required
    if seperate_tracking_res:
        tracking_dataset = get_dataset(
            config_dict=gradslam_data_cfg,
            basedir=dataset_config["basedir"],
            sequence=os.path.basename(dataset_config["sequence"]),
            start=dataset_config["start"],
            end=dataset_config["end"],
            stride=dataset_config["stride"],
            desired_height=dataset_config["tracking_image_height"],
            desired_width=dataset_config["tracking_image_width"],
            device=device,
            relative_pose=True,
            preload = dataset_config["preload"],
            ignore_bad=dataset_config["ignore_bad"],
            use_train_split=dataset_config["use_train_split"],
            train_or_test=dataset_config["train_or_test"]
        )
        tracking_color, _, tracking_intrinsics, _ = tracking_dataset[0]
        tracking_color = tracking_color.permute(2, 0, 1) / 255 # (H, W, C) -> (C, H, W)
        tracking_intrinsics = tracking_intrinsics[:3, :3]
        tracking_cam = setup_camera(tracking_color.shape[2], tracking_color.shape[1], 
                                    tracking_intrinsics.cpu().numpy(), first_frame_w2c.detach().cpu().numpy(), 
                                    use_simplification=config['gaussian_simplification'])
    
    # Initialize list to keep track of Keyframes
    keyframe_list = []
    keyframe_time_indices = []
    
    # Init Variables to keep track of ground truth poses and runtimes
    gt_w2c_all_frames = []
    tracking_iter_time_sum = 0
    tracking_iter_time_count = 0
    mapping_iter_time_sum = 0
    mapping_iter_time_count = 0
    tracking_frame_time_sum = 0
    tracking_frame_time_count = 0
    mapping_frame_time_sum = 0
    mapping_frame_time_count = 0

    # Load Checkpoint
    if config['load_checkpoint']:
        checkpoint_time_idx = config['checkpoint_time_idx']
        print(f"Loading Checkpoint for Frame {checkpoint_time_idx}")
        ckpt_path = os.path.join(config['workdir'], config['run_name'], f"params{checkpoint_time_idx}.npz")
        params = dict(np.load(ckpt_path, allow_pickle=True))
        params = {k: torch.tensor(params[k]).cuda().float().requires_grad_(True) for k in params.keys()}
        variables['max_2D_radius'] = torch.zeros(params['means3D'].shape[0]).cuda().float()
        variables['means2D_gradient_accum'] = torch.zeros(params['means3D'].shape[0]).cuda().float()
        variables['denom'] = torch.zeros(params['means3D'].shape[0]).cuda().float()
        variables['timestep'] = torch.zeros(params['means3D'].shape[0]).cuda().float()
        # Load the keyframe time idx list
        keyframe_time_indices = np.load(os.path.join(config['workdir'], config['run_name'], f"keyframe_time_indices{checkpoint_time_idx}.npy"))
        keyframe_time_indices = keyframe_time_indices.tolist()
        # Update the ground truth poses list
        for time_idx in range(checkpoint_time_idx):
            # Load RGBD frames incrementally instead of all frames
            color, depth, _, gt_pose = dataset[time_idx]
            # Process poses
            gt_w2c = torch.linalg.inv(gt_pose)
            gt_w2c_all_frames.append(gt_w2c)
            # Initialize Keyframe List
            if time_idx in keyframe_time_indices:
                # Get the estimated rotation & translation
                curr_cam_rot = F.normalize(params['cam_unnorm_rots'][..., time_idx].detach())
                curr_cam_tran = params['cam_trans'][..., time_idx].detach()
                curr_w2c = torch.eye(4).cuda().float()
                curr_w2c[:3, :3] = build_rotation(curr_cam_rot)
                curr_w2c[:3, 3] = curr_cam_tran
                # Initialize Keyframe Info
                color = color.permute(2, 0, 1) / 255
                depth = depth.permute(2, 0, 1)
                curr_keyframe = {'id': time_idx, 'est_w2c': curr_w2c, 'color': color, 'depth': depth}
                # Add to keyframe list
                keyframe_list.append(curr_keyframe)
    else:
        checkpoint_time_idx = 0
    
    # timer.lap("all the config")
    
    # Iterate over Scan
    for time_idx in tqdm(range(checkpoint_time_idx, num_frames)):
        
        # timer.lap("iterating over frame "+str(time_idx), 0)
        
        print() # always show global iteration
        # Load RGBD frames incrementally instead of all frames
        color, depth, _, gt_pose = dataset[time_idx]
        # Process poses
        gt_w2c = torch.linalg.inv(gt_pose)
        # Process RGB-D Data
        color = color.permute(2, 0, 1) / 255
        depth = depth.permute(2, 0, 1)
        gt_w2c_all_frames.append(gt_w2c)
        curr_gt_w2c = gt_w2c_all_frames
        # Optimize only current time step for tracking
        iter_time_idx = time_idx
        # Initialize Mapping Data for selected frame
        curr_data = {'cam': cam, 'im': color, 'depth': depth, 'id': iter_time_idx, 'intrinsics': intrinsics, 
                     'w2c': first_frame_w2c, 'iter_gt_w2c_list': curr_gt_w2c}
        
        # Initialize Data for Tracking
        if seperate_tracking_res:
            tracking_color, tracking_depth, _, _ = tracking_dataset[time_idx]
            tracking_color = tracking_color.permute(2, 0, 1) / 255
            tracking_depth = tracking_depth.permute(2, 0, 1)
            tracking_curr_data = {'cam': tracking_cam, 'im': tracking_color, 'depth': tracking_depth, 'id': iter_time_idx,
                                  'intrinsics': tracking_intrinsics, 'w2c': first_frame_w2c, 'iter_gt_w2c_list': curr_gt_w2c}
        else:
            tracking_curr_data = curr_data

        # Optimization Iterations
        num_iters_mapping = config['mapping']['num_iters']
        
        # Initialize the camera pose for the current frame
        if time_idx > 0:
            params = initialize_camera_pose(params, time_idx, forward_prop=config['tracking']['forward_prop'])

        # timer.lap("initialized data", 1)

        # Tracking
        tracking_start_time = time.time()
        if time_idx > 0 and not config['tracking']['use_gt_poses']:
            # Reset Optimizer & Learning Rates for tracking
            optimizer = initialize_optimizer(params, config['tracking']['lrs'])
            # Keep Track of Best Candidate Rotation & Translation
            candidate_cam_unnorm_rot = params['cam_unnorm_rots'][..., time_idx].detach().clone()
            candidate_cam_tran = params['cam_trans'][..., time_idx].detach().clone()
            current_min_loss = float(1e20)
            # Tracking Optimization
            iter = 0
            do_continue_slam = False
            num_iters_tracking = config['tracking']['num_iters']
            progress_bar = tqdm(range(num_iters_tracking), desc=f"Tracking Time Step: {time_idx}")
            while True:
                iter_start_time = time.time()
                # Loss for current frame
                loss, variables, losses = get_loss(params, tracking_curr_data, variables, iter_time_idx, config['tracking']['loss_weights'],
                                                   config['tracking']['use_sil_for_loss'], config['tracking']['sil_thres'],
                                                   config['tracking']['use_l1'], config['tracking']['ignore_outlier_depth_loss'], tracking=True, 
                                                   plot_dir=eval_dir, visualize_tracking_loss=config['tracking']['visualize_tracking_loss'],
                                                   tracking_iteration=iter)
                # Backprop
                loss.backward()
                # Optimizer Update
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                with torch.no_grad():
                    # Save the best candidate rotation & translation
                    if loss < current_min_loss:
                        current_min_loss = loss
                        candidate_cam_unnorm_rot = params['cam_unnorm_rots'][..., time_idx].detach().clone()
                        candidate_cam_tran = params['cam_trans'][..., time_idx].detach().clone()
                    # Report Progress
                    if config['report_iter_progress']:
                        report_progress(params, tracking_curr_data, iter+1, progress_bar, iter_time_idx, sil_thres=config['tracking']['sil_thres'], tracking=True)
                    else:
                        progress_bar.update(1)
                # Update the runtime numbers
                iter_end_time = time.time()
                tracking_iter_time_sum += iter_end_time - iter_start_time
                tracking_iter_time_count += 1
                # Check if we should stop tracking
                iter += 1
                if iter == num_iters_tracking:
                    if losses['depth'] < config['tracking']['depth_loss_thres'] and config['tracking']['use_depth_loss_thres']:
                        break
                    elif config['tracking']['use_depth_loss_thres'] and not do_continue_slam:
                        do_continue_slam = True
                        progress_bar = tqdm(range(num_iters_tracking), desc=f"Tracking Time Step: {time_idx}")
                        num_iters_tracking = 2*num_iters_tracking
                    else:
                        break

            progress_bar.close()
            # Copy over the best candidate rotation & translation
            with torch.no_grad():
                params['cam_unnorm_rots'][..., time_idx] = candidate_cam_unnorm_rot
                params['cam_trans'][..., time_idx] = candidate_cam_tran
        elif time_idx > 0 and config['tracking']['use_gt_poses']:
            with torch.no_grad():
                # Get the ground truth pose relative to frame 0
                rel_w2c = curr_gt_w2c[-1]
                rel_w2c_rot = rel_w2c[:3, :3].unsqueeze(0).detach()
                rel_w2c_rot_quat = matrix_to_quaternion(rel_w2c_rot)
                rel_w2c_tran = rel_w2c[:3, 3].detach()
                # Update the camera parameters
                params['cam_unnorm_rots'][..., time_idx] = rel_w2c_rot_quat
                params['cam_trans'][..., time_idx] = rel_w2c_tran
        # Update the runtime numbers
        tracking_end_time = time.time()
        tracking_frame_time_sum += tracking_end_time - tracking_start_time
        tracking_frame_time_count += 1

        # timer.lap("tracking done", 2)

        # Densification & KeyFrame-based Mapping
        if time_idx == 0 or (time_idx+1) % config['map_every'] == 0:
            # Densification
            if config['mapping']['add_new_gaussians'] and time_idx > 0:
                # Setup Data for Densification
                if seperate_densification_res:
                    # Load RGBD frames incrementally instead of all frames
                    densify_color, densify_depth, _, _ = densify_dataset[time_idx]
                    densify_color = densify_color.permute(2, 0, 1) / 255
                    densify_depth = densify_depth.permute(2, 0, 1)
                    densify_curr_data = {'cam': densify_cam, 'im': densify_color, 'depth': densify_depth, 'id': time_idx, 
                                 'intrinsics': densify_intrinsics, 'w2c': first_frame_w2c, 'iter_gt_w2c_list': curr_gt_w2c}
                else:
                    densify_curr_data = curr_data

                # delete floating gaussians
                # params, variables = remove_floating_gaussians(params, variables, densify_curr_data, time_idx)
                
                # Add new Gaussians to the scene based on the Silhouette
                params, variables = add_new_gaussians(params, variables, densify_curr_data, 
                                                      config['mapping']['sil_thres'], time_idx,
                                                      config['mean_sq_dist_method'], 
                                                      config['gaussian_simplification'])
                post_num_pts = params['means3D'].shape[0]
            
            if not config['distance_keyframe_selection']:
                with torch.no_grad():
                    # Get the current estimated rotation & translation
                    curr_cam_rot = F.normalize(params['cam_unnorm_rots'][..., time_idx].detach())
                    curr_cam_tran = params['cam_trans'][..., time_idx].detach()
                    curr_w2c = torch.eye(4).cuda().float()
                    curr_w2c[:3, :3] = build_rotation(curr_cam_rot)
                    curr_w2c[:3, 3] = curr_cam_tran
                    # Select Keyframes for Mapping
                    num_keyframes = config['mapping_window_size']-2
                    selected_keyframes = keyframe_selection_overlap(depth, curr_w2c, intrinsics, keyframe_list[:-1], num_keyframes)
                    selected_time_idx = [keyframe_list[frame_idx]['id'] for frame_idx in selected_keyframes]
                    if len(keyframe_list) > 0:
                        # Add last keyframe to the selected keyframes
                        selected_time_idx.append(keyframe_list[-1]['id'])
                        selected_keyframes.append(len(keyframe_list)-1)
                    # Add current frame to the selected keyframes
                    selected_time_idx.append(time_idx)
                    selected_keyframes.append(-1)
                    # Print the selected keyframes
                    print(f"\nSelected Keyframes at Frame {time_idx}: {selected_time_idx}")

            # Reset Optimizer & Learning Rates for Full Map Optimization
            optimizer = initialize_optimizer(params, config['mapping']['lrs']) 

            # timer.lap("Densification Done at frame "+str(time_idx), 3)

            # Mapping
            mapping_start_time = time.time()
            if num_iters_mapping > 0:
                progress_bar = tqdm(range(num_iters_mapping), desc=f"Mapping Time Step: {time_idx}")
                
            actural_keyframe_ids = []
            for iter in range(num_iters_mapping):
                iter_start_time = time.time()
                if not config['distance_keyframe_selection']:
                    # Randomly select a frame until current time step amongst keyframes
                    rand_idx = np.random.randint(0, len(selected_keyframes))
                    selected_rand_keyframe_idx = selected_keyframes[rand_idx]
                    actural_keyframe_ids.append(selected_rand_keyframe_idx)
                    if selected_rand_keyframe_idx == -1:
                        # Use Current Frame Data
                        iter_time_idx = time_idx
                        iter_color = color
                        iter_depth = depth
                    else:
                        # Use Keyframe Data
                        iter_time_idx = keyframe_list[selected_rand_keyframe_idx]['id']
                        iter_color = keyframe_list[selected_rand_keyframe_idx]['color']
                        iter_depth = keyframe_list[selected_rand_keyframe_idx]['depth']
                else:
                    if len(actural_keyframe_ids) == 0:
                        if len(keyframe_list) > 0:
                            curr_position = params['cam_trans'][..., time_idx].detach().cpu()
                            actural_keyframe_ids = keyframe_selection_distance(time_idx, curr_position, keyframe_list, config['distance_current_frame_prob'], num_iters_mapping)
                        else:
                            actural_keyframe_ids = [0] * num_iters_mapping
                        print(f"\nUsed Frames for mapping at Frame {time_idx}: {[keyframe_list[i]['id'] if i != len(keyframe_list) else 'curr' for i in actural_keyframe_ids]}")

                    selected_keyframe_ids = actural_keyframe_ids[iter]

                    if selected_keyframe_ids == len(keyframe_list):
                        # Use Current Frame Data
                        iter_time_idx = time_idx
                        iter_color = color
                        iter_depth = depth
                    else:
                        # Use Keyframe Data
                        iter_time_idx = keyframe_list[selected_keyframe_ids]['id']
                        iter_color = keyframe_list[selected_keyframe_ids]['color']
                        iter_depth = keyframe_list[selected_keyframe_ids]['depth']
                    
                    
                iter_gt_w2c = gt_w2c_all_frames[:iter_time_idx+1]
                iter_data = {'cam': cam, 'im': iter_color, 'depth': iter_depth, 'id': iter_time_idx, 
                             'intrinsics': intrinsics, 'w2c': first_frame_w2c, 'iter_gt_w2c_list': iter_gt_w2c}
                # Loss for current frame
                loss, variables, losses = get_loss(params, iter_data, variables, iter_time_idx, config['mapping']['loss_weights'],
                                                config['mapping']['use_sil_for_loss'], config['mapping']['sil_thres'],
                                                config['mapping']['use_l1'], config['mapping']['ignore_outlier_depth_loss'], mapping=True)
                # Backprop
                loss.backward()
                with torch.no_grad():
                    # Prune Gaussians (Innovation 1: enhanced pruning)
                    if config['mapping']['prune_gaussians']:
                        innovation_cfg = config.get('innovations', {})
                        prune_transformed_pts = None
                        if innovation_cfg.get('enable_visibility_pruning', False):
                            with torch.no_grad():
                                prune_transformed_pts = transform_to_frame(params, iter_time_idx, gaussians_grad=False, camera_grad=False)
                        params, variables = prune_gaussians(params, variables, optimizer, iter, config['mapping']['pruning_dict'],
                                                            innovation_config=innovation_cfg, curr_data=iter_data,
                                                            transformed_pts=prune_transformed_pts)
                    # Gaussian-Splatting's Gradient-based Densification
                    if config['mapping']['use_gaussian_splatting_densification']:
                        params, variables = densify(params, variables, optimizer, iter, config['mapping']['densify_dict'])
                    # Optimizer Update
                    optimizer.step()
                    optimizer.zero_grad(set_to_none=True)

                    # === Innovation 1+3: Update three-way visibility classifier ===
                    innovation_cfg = config.get('innovations', {})
                    if (innovation_cfg.get('enable_visibility_pruning', False) and 'gauss_vis' in variables):
                        variables = update_three_way_classifier(variables, variables['gauss_vis'], innovation_cfg)

                    # === Innovation 3: Store previous deformation offsets for temporal regularization ===
                    if innovation_cfg.get('enable_deformation', False) and 'deform_offsets' in params:
                        variables['prev_deform_offsets'] = params['deform_offsets'].detach().clone()

                    # Report Progress
                    if config['report_iter_progress']:
                        report_progress(params, iter_data, iter+1, progress_bar, iter_time_idx, sil_thres=config['mapping']['sil_thres'], 
                                        mapping=True, online_time_idx=time_idx)
                    else:
                        progress_bar.update(1)
                # Update the runtime numbers
                iter_end_time = time.time()
                mapping_iter_time_sum += iter_end_time - iter_start_time
                mapping_iter_time_count += 1
            if num_iters_mapping > 0:
                progress_bar.close()
            # Update the runtime numbers
            mapping_end_time = time.time()
            mapping_frame_time_sum += mapping_end_time - mapping_start_time
            mapping_frame_time_count += 1

            # === Innovation 2: Periodic Bundle Adjustment ===
            innovation_cfg = config.get('innovations', {})
            if (innovation_cfg.get('enable_periodic_ba', False) and
                time_idx > 0 and 
                (time_idx + 1) % innovation_cfg.get('ba_every_m_frames', 20) == 0 and
                len(keyframe_list) >= 2):
                print(f"\n--- Periodic BA at frame {time_idx} ---")
                params, variables = periodic_bundle_adjustment(
                    params, variables, keyframe_list, config, intrinsics, cam, first_frame_w2c)

            if time_idx == 0 or (time_idx+1) % config['report_global_progress_every'] == 0:
                try:
                    # Report Mapping Progress
                    progress_bar = tqdm(range(1), desc=f"Mapping Result Time Step: {time_idx}")
                    with torch.no_grad():
                        report_progress(params, curr_data, 1, progress_bar, time_idx, sil_thres=config['mapping']['sil_thres'], 
                                        mapping=True, online_time_idx=time_idx)
                    progress_bar.close()
                except:
                    ckpt_output_dir = os.path.join(config["workdir"], config["run_name"])
                    save_params_ckpt(params, ckpt_output_dir, time_idx)
                    print('Failed to evaluate trajectory.')
        
        # timer.lap('Mapping Done.', 4)
        
        # Add frame to keyframe list
        if ((time_idx == 0) or ((time_idx+1) % config['keyframe_every'] == 0) or \
                    (time_idx == num_frames-2)) and (not torch.isinf(curr_gt_w2c[-1]).any()) and (not torch.isnan(curr_gt_w2c[-1]).any()):
            with torch.no_grad():
                # Get the current estimated rotation & translation
                curr_cam_rot = F.normalize(params['cam_unnorm_rots'][..., time_idx].detach())
                curr_cam_tran = params['cam_trans'][..., time_idx].detach()
                curr_w2c = torch.eye(4).cuda().float()
                curr_w2c[:3, :3] = build_rotation(curr_cam_rot)
                curr_w2c[:3, 3] = curr_cam_tran
                # Initialize Keyframe Info
                curr_keyframe = {'id': time_idx, 'est_w2c': curr_w2c, 'color': color, 'depth': depth}
                # Add to keyframe list
                keyframe_list.append(curr_keyframe)
                keyframe_time_indices.append(time_idx)
        
        # Checkpoint every iteration
        if time_idx % config["checkpoint_interval"] == 0 and config['save_checkpoints']:
            ckpt_output_dir = os.path.join(config["workdir"], config["run_name"])
            save_params_ckpt(params, ckpt_output_dir, time_idx)
            np.save(os.path.join(ckpt_output_dir, f"keyframe_time_indices{time_idx}.npy"), np.array(keyframe_time_indices))
        

        torch.cuda.empty_cache()

    # timer.end()

    # Compute Average Runtimes
    if tracking_iter_time_count == 0:
        tracking_iter_time_count = 1
        tracking_frame_time_count = 1
    if mapping_iter_time_count == 0:
        mapping_iter_time_count = 1
        mapping_frame_time_count = 1
    tracking_iter_time_avg = tracking_iter_time_sum / tracking_iter_time_count
    tracking_frame_time_avg = tracking_frame_time_sum / tracking_frame_time_count
    mapping_iter_time_avg = mapping_iter_time_sum / mapping_iter_time_count
    mapping_frame_time_avg = mapping_frame_time_sum / mapping_frame_time_count
    print(f"\nAverage Tracking/Iteration Time: {tracking_iter_time_avg*1000} ms")
    print(f"Average Tracking/Frame Time: {tracking_frame_time_avg} s")
    print(f"Average Mapping/Iteration Time: {mapping_iter_time_avg*1000} ms")
    print(f"Average Mapping/Frame Time: {mapping_frame_time_avg} s")
    with open(os.path.join(output_dir, "runtimes.txt"), "w") as f:
        f.write(f"Average Tracking/Iteration Time: {tracking_iter_time_avg*1000} ms\n")
        f.write(f"Average Tracking/Frame Time: {tracking_frame_time_avg} s\n")
        f.write(f"Average Mapping/Iteration Time: {mapping_iter_time_avg*1000} ms\n")
        f.write(f"Average Mapping/Frame Time: {mapping_frame_time_avg} s\n")
        f.write(f"Frame Time: {tracking_frame_time_avg + mapping_frame_time_avg} s\n")
    
    # Evaluate Final Parameters
    dataset = [dataset, eval_dataset, 'C3VD'] if dataset_config["train_or_test"] == 'train' else dataset
    with torch.no_grad():
        eval_save(dataset, params, eval_dir, sil_thres=config['mapping']['sil_thres'],
                mapping_iters=config['mapping']['num_iters'], add_new_gaussians=config['mapping']['add_new_gaussians'])

    # Add Camera Parameters to Save them
    params['timestep'] = variables['timestep']
    params['intrinsics'] = intrinsics.detach().cpu().numpy()
    params['w2c'] = first_frame_w2c.detach().cpu().numpy()
    params['org_width'] = dataset_config["desired_image_width"]
    params['org_height'] = dataset_config["desired_image_height"]
    params['gt_w2c_all_frames'] = []
    for gt_w2c_tensor in gt_w2c_all_frames:
        params['gt_w2c_all_frames'].append(gt_w2c_tensor.detach().cpu().numpy())
    params['gt_w2c_all_frames'] = np.stack(params['gt_w2c_all_frames'], axis=0)
    params['keyframe_time_indices'] = np.array(keyframe_time_indices)
    
    # Save Parameters
    save_params(params, output_dir)
    save_means3D(params['means3D'], output_dir)

if __name__ == "__main__":
    parser = argparse.ArgumentParser()

    parser.add_argument("experiment", type=str, help="Path to experiment file")
    # parser.add_argument("--online_vis", action="store_true", help="Visualize mapping renderings while running")

    args = parser.parse_args()

    experiment = SourceFileLoader(
        os.path.basename(args.experiment), args.experiment
    ).load_module()

    # Prepare dir for visualization
    # if args.online_vis:
    #     vis_dir = './online_vis'
    #     os.makedirs(vis_dir, exist_ok=True)
    #     for filename in os.listdir(vis_dir):
    #         os.unlink(os.path.join(vis_dir, filename))

    # Set Experiment Seed
    seed_everything(seed=experiment.config['seed'])
    
    # Create Results Directory and Copy Config
    results_dir = os.path.join(
        experiment.config["workdir"], experiment.config["run_name"]
    )
    if not experiment.config['load_checkpoint']:
        os.makedirs(results_dir, exist_ok=True)
        shutil.copy(args.experiment, os.path.join(results_dir, "config.py"))

    rgbd_slam(experiment.config)
    
    plot_video(os.path.join(results_dir, 'eval', 'plots'), os.path.join('./experiments/', experiment.group_name, experiment.scene_name, 'keyframes'))

5. Run Training

In [ ]:
%cd /content/EndoGSLAM
!python scripts/main.py configs/c3vd/c3vd_innovations.py

In [ ]:
%cd /content/EndoGSLAM
!python scripts/calc_metrics.py --gt data/C3VD/sigmoid_t3_a --render experiments/C3VD_innovations/sigmoid_t3_a --test_single

6. Run All Sequences

In [ ]:
import os, re
%cd /content/EndoGSLAM

sequences = [
    'cecum_t1_b',
    'cecum_t2_b',
    'cecum_t3_a',
    'sigmoid_t1_a',
    'sigmoid_t2_a',
    'sigmoid_t3_a',
    'trans_t1_b',
    'trans_t2_c',
    'trans_t4_a',
    'trans_t4_b',
]

all_results = {}

for i, seq in enumerate(sequences):
    print(f"\n{'='*60}")
    print(f"Processing [{i+1}/10] {seq}")
    print('='*60)

    # Set SCENE_NUM env variable and run training
    os.environ['SCENE_NUM'] = str(i)
    !SCENE_NUM={i} python scripts/main.py configs/c3vd/c3vd_innovations.py 2>&1 | grep -v "it/s\]"

    # Evaluate
    !python scripts/calc_metrics.py --gt data/C3VD/{seq} --render experiments/C3VD_innovations/{seq} --test_single

    # Extract FPS from runtimes.txt
    rt_path = f'experiments/C3VD_innovations/{seq}/runtimes.txt'
    fps = 0.0
    if os.path.exists(rt_path):
        with open(rt_path, 'r') as f:
            for line in f:
                if 'Frame Time:' in line:
                    m = re.search(r'Frame Time:\s*([\d.]+)', line)
                    if m:
                        frame_time = float(m.group(1))
                        fps = 1.0 / frame_time if frame_time > 0 else 0.0
        print(f'FPS: {fps:.2f}')
    all_results[seq] = {'fps': fps}

# === Print Summary Table ===
print(f"\n\n{'='*60}")
print('SUMMARY: All Sequences')
print('='*60)
total_fps = 0
for seq, res in all_results.items():
    print(f"{seq:20s} | FPS: {res['fps']:.2f}")
    total_fps += res['fps']
avg_fps = total_fps / len(all_results) if all_results else 0
print(f"{'AVERAGE':20s} | FPS: {avg_fps:.2f}")